# 01 - POC Data Generation

**Purpose:** Generate synthetic data for POC testing. NOT used in production.

**Run:** Once for POC setup, or when refreshing test data.

**Runtime:** ~1-2 minutes (depends on scale)

In [1]:
# 1. Imports
from snowflake.snowpark import Session
import pandas as pd
import numpy as np
from datetime import date, timedelta
from utils import get_connection_config, get_poc_data_config

conn_cfg = get_connection_config()
poc_cfg = get_poc_data_config()

session = Session.builder.getOrCreate()
session.sql(f"USE DATABASE {conn_cfg['database']}").collect()
session.sql(f"USE SCHEMA {conn_cfg['schema_data']}").collect()
session.use_warehouse(conn_cfg['warehouse'])

print(f"Connected: {session.get_current_account()}")

from modin.config import AutoSwitchBackend; AutoSwitchBackend.disable()


Connected: "SFSENORTHAMERICA-AFERAS_AWS1"


In [2]:
# 2. Configuration (loaded from config.yaml)
NUM_PARTITIONS = poc_cfg["partitions"]
DAYS_HISTORY = poc_cfg["days_history"]
START_DATE = poc_cfg["start_date"]

print(f"POC Data Configuration:")
print(f"   Partitions: {NUM_PARTITIONS}")
print(f"   History:    {DAYS_HISTORY} days")
print(f"   Rows:       ~{NUM_PARTITIONS * DAYS_HISTORY:,}")

POC Data Configuration:
   Partitions: 100
   History:    730 days
   Rows:       ~73,000


In [3]:
# 3. Generate FEATURE_TABLE
print("Generating feature data with realistic patterns...")
np.random.seed(42)

records = []
dates = [START_DATE + timedelta(days=i) for i in range(DAYS_HISTORY)]

for partition_idx in range(NUM_PARTITIONS):
    partition_id = f"P{str(partition_idx).zfill(4)}"
    
    base_target = np.random.uniform(20, 100)
    partition_effect = np.random.uniform(0.7, 1.3)
    
    for d in dates:
        doy = d.timetuple().tm_yday
        dow = d.weekday()
        
        seasonal = 1.0 + 0.2 * np.sin(2 * np.pi * doy / 365)
        weekday_effect = 1.1 if dow in (4, 5) else 1.0
        
        feature_1 = np.random.normal(50, 10) * partition_effect
        feature_2 = np.random.normal(30, 5) * seasonal
        feature_3 = np.random.uniform(0, 100)
        feature_4 = doy / 365.0
        feature_5 = dow / 6.0
        
        noise = np.random.normal(0, 3)
        target = (base_target * partition_effect * seasonal * weekday_effect 
                  + 0.3 * feature_1 + 0.2 * feature_2 + 0.1 * feature_3 + noise)
        target = max(0, round(target, 2))
        
        records.append({
            "ACTUAL_DATE": d,
            "PARTITION_ID": partition_id,
            "FEATURE_1": round(feature_1, 4),
            "FEATURE_2": round(feature_2, 4),
            "FEATURE_3": round(feature_3, 4),
            "FEATURE_4": round(feature_4, 4),
            "FEATURE_5": round(feature_5, 4),
            "TARGET": target,
        })

print(f"Generated {len(records):,} rows")

Generating feature data with realistic patterns...
Generated 73,000 rows


In [4]:
# 4. Upload FEATURE_TABLE in chunks
print("Uploading to FEATURE_TABLE...")

for i in range(0, len(records), 100_000):
    chunk = pd.DataFrame(records[i : i + 100_000])
    session.write_pandas(chunk, "FEATURE_TABLE", overwrite=(i == 0))
    print(f"  Uploaded {min(i + 100_000, len(records)):,} / {len(records):,}")

print(f"FEATURE_TABLE ready with {len(records):,} rows")

Uploading to FEATURE_TABLE...
  Uploaded 73,000 / 73,000
FEATURE_TABLE ready with 73,000 rows


In [5]:
# 5. Verify Data
print("\n" + "="*60)
print("POC DATA GENERATION COMPLETE")
print("="*60)

counts = session.sql("""
    SELECT 
        COUNT(*) AS TOTAL_ROWS,
        COUNT(DISTINCT PARTITION_ID) AS PARTITIONS,
        MIN(ACTUAL_DATE) AS MIN_DATE,
        MAX(ACTUAL_DATE) AS MAX_DATE
    FROM FEATURE_TABLE
""").collect()[0]

print(f"\nData Summary:")
print(f"  Total rows:  {counts['TOTAL_ROWS']:,}")
print(f"  Partitions:  {counts['PARTITIONS']:,}")
print(f"  Date range:  {counts['MIN_DATE']} to {counts['MAX_DATE']}")


POC DATA GENERATION COMPLETE

Data Summary:
  Total rows:  73,000
  Partitions:  100
  Date range:  2024-03-02 to 2026-03-01
